# 07 Prepare Manuscript Tables and Figure Data

Purpose: turn the reviewed comparative evidence from notebook 06 into manuscript-facing tables and clean figure datasets.

This notebook does not search, download, or extract sources. It uses only reviewed and synthesised outputs from notebook 06.


In [ ]:
from pathlib import Path
from datetime import date
import re
import math
import pandas as pd
from copy import copy

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

RUN_DATE = date.today().isoformat()


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    marker = Path('Research_Workflow/03_Synthesis/comparative_evidence/comparative_evidence_master_table.csv')
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
        nested = p / 'DC_job_potential'
        if (nested / marker).exists():
            return nested
    raise FileNotFoundError('Could not locate DC_job_potential project root from current working directory.')

PROJECT_ROOT = find_project_root()
SYNTHESIS_IN = PROJECT_ROOT / 'Research_Workflow' / '03_Synthesis' / 'comparative_evidence'
OUTPUT_DIR = PROJECT_ROOT / 'Research_Workflow' / '03_Synthesis' / 'manuscript_tables_figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Input dir:', SYNTHESIS_IN)
print('Output dir:', OUTPUT_DIR)


In [ ]:
inputs = {
    'master': SYNTHESIS_IN / 'comparative_evidence_master_table.csv',
    'primary': SYNTHESIS_IN / 'comparative_evidence_primary_table.csv',
    'jobs': SYNTHESIS_IN / 'jobs_intensity_comparison_for_figures.csv',
    'scale': SYNTHESIS_IN / 'scale_context_table.csv',
    'gap_log': SYNTHESIS_IN / 'evidence_gap_log.csv',
}
missing = [name for name, path in inputs.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing notebook 06 output(s): ' + ', '.join(missing))

master = pd.read_csv(inputs['master'], dtype=str, keep_default_na=False)
primary = pd.read_csv(inputs['primary'], dtype=str, keep_default_na=False)
jobs = pd.read_csv(inputs['jobs'], dtype=str, keep_default_na=False)
scale = pd.read_csv(inputs['scale'], dtype=str, keep_default_na=False)
gap_log = pd.read_csv(inputs['gap_log'], dtype=str, keep_default_na=False)

print('master', master.shape)
print('primary', primary.shape)
print('jobs candidates', jobs.shape)
print('scale context', scale.shape)
print('gap log', gap_log.shape)

assert master['evidence_id'].is_unique, 'Run notebook 06 again: evidence_id is not unique.'


In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()


def short_text(value, limit=280):
    text = clean_text(value)
    if len(text) <= limit:
        return text
    return text[:limit-3].rstrip() + '...'


def source_label(row):
    title = clean_text(row.get('source_title', ''))
    sid = clean_text(row.get('source_id', ''))
    return f'{title} ({sid})' if sid else title


def domain_label(domain):
    return 'Data centre' if domain == 'data_centre' else 'Comparator asset'


def manuscript_use(row):
    role = clean_text(row.get('evidence_role', ''))
    rel = clean_text(row.get('comparison_relevance', ''))
    if role == 'primary_citable_after_spotcheck':
        return 'Primary quantitative evidence after source spot-check'
    if role == 'supporting_with_caveats':
        return 'Use with caveats in notes'
    if rel == 'scale_context_not_job_creation':
        return 'Scale/context only; do not infer jobs'
    if role == 'context_only':
        return 'Narrative context only'
    return 'Reviewed evidence; inspect notes before use'


def caveat_summary(row):
    bits = []
    direct = clean_text(row.get('direct_vs_total', ''))
    projected = clean_text(row.get('projected_vs_realised', ''))
    notes = clean_text(row.get('notes', ''))
    if direct:
        bits.append(f'Scope: {direct}')
    if projected:
        bits.append(f'Basis: {projected}')
    if notes:
        bits.append(short_text(notes, 180))
    return ' | '.join(bits)

# Manuscript Table 1: compact, but still audit-ready.
table1 = primary.copy()
table1_out = pd.DataFrame({
    'table_row_id': [f't1_{i:03d}' for i in range(1, len(table1) + 1)],
    'evidence_id': table1['evidence_id'],
    'domain': table1['domain'].map(domain_label),
    'asset_or_comparator': table1['asset_class'],
    'phase': table1['phase'],
    'metric_family': table1['metric_family'],
    'verified_value': table1['verified_value'],
    'metric_definition': table1['verified_metric_definition'].map(lambda x: short_text(x, 320)),
    'direct_vs_total': table1['direct_vs_total'],
    'projected_vs_realised': table1['projected_vs_realised'],
    'geography': table1['geography'],
    'source': table1.apply(source_label, axis=1),
    'source_page_or_location': table1['page_or_location'],
    'review_decision': table1['review_decision'],
    'evidence_role': table1['evidence_role'],
    'manuscript_use': table1.apply(manuscript_use, axis=1),
    'caveat_summary': table1.apply(caveat_summary, axis=1),
    'source_url': table1['source_url'],
    'source_raw_path': table1['source_raw_path'],
    'spotcheck_required': table1['requires_source_spotcheck'],
})

print('Table 1 rows:', len(table1_out))
display(table1_out.head(10))


In [ ]:
def parse_number(text):
    text = clean_text(text).replace(',', '')
    match = re.search(r'(?<![A-Za-z])([0-9]+(?:\.[0-9]+)?)', text)
    return float(match.group(1)) if match else None


def parse_range(text):
    text = clean_text(text).replace(',', '')
    text = text.replace('–', '-').replace('—', '-')
    match = re.search(r'([0-9]+(?:\.[0-9]+)?)\s*-\s*([0-9]+(?:\.[0-9]+)?)', text)
    if match:
        low = float(match.group(1))
        high = float(match.group(2))
        return low, high, (low + high) / 2
    return None


def first_numeric_or_none(text):
    value = parse_number(text)
    return None if value is None else float(value)


def figure_panel(metric_family, unit):
    if unit in ['jobs_per_mw', 'operational_fte_per_mw', 'job_years_per_mw']:
        return 'A. Jobs or job-years per MW'
    if unit == 'jobs_per_usd_million':
        return 'B. Jobs per USD 1 million investment/spending'
    if unit == 'job_years_per_gwh':
        return 'C. Job-years per GWh'
    if unit == 'percent_effect':
        return 'D. Causal/local percentage effects'
    return 'E. Absolute/context benchmarks'


def base_fig_row(row):
    return {
        'source_evidence_id': clean_text(row.get('evidence_id', '')),
        'review_id': clean_text(row.get('review_id', '')),
        'domain': clean_text(row.get('domain', '')),
        'domain_label': domain_label(clean_text(row.get('domain', ''))),
        'asset_class': clean_text(row.get('asset_class', '')),
        'source_id': clean_text(row.get('source_id', '')),
        'source_title': clean_text(row.get('source_title', '')),
        'phase': clean_text(row.get('phase', '')),
        'metric_family': clean_text(row.get('metric_family', '')),
        'direct_vs_total': clean_text(row.get('direct_vs_total', '')),
        'projected_vs_realised': clean_text(row.get('projected_vs_realised', '')),
        'geography': clean_text(row.get('geography', '')),
        'review_decision': clean_text(row.get('review_decision', '')),
        'evidence_role': clean_text(row.get('evidence_role', '')),
        'source_url': clean_text(row.get('source_url', '')),
        'source_raw_path': clean_text(row.get('source_raw_path', '')),
        'source_page_or_location': clean_text(row.get('page_or_location', '')),
        'original_verified_value': clean_text(row.get('verified_value', '')),
        'metric_definition': clean_text(row.get('verified_metric_definition', '')),
        'notes': clean_text(row.get('notes', '')),
    }


def add_fig_row(rows, row, unit, label, value=None, value_low=None, value_high=None, split_method='automatic_numeric', derived=False, plot_ready=True, figure_use='plot_candidate'):
    base = base_fig_row(row)
    if value is None and value_low is not None and value_high is not None:
        value = (value_low + value_high) / 2
    if unit == 'operational_fte_per_mw':
        base['phase'] = 'operation'
    panel = figure_panel(base['metric_family'], unit)
    base.update({
        'figure_row_id': '',
        'figure_panel': panel,
        'figure_unit': unit,
        'series_label': label,
        'plot_value': value,
        'plot_value_low': value_low if value_low is not None else value,
        'plot_value_high': value_high if value_high is not None else value,
        'range_flag': value_low is not None and value_high is not None and value_low != value_high,
        'derived_flag': bool(derived),
        'split_method': split_method,
        'plot_ready': bool(plot_ready),
        'figure_use': figure_use,
    })
    rows.append(base)


def split_job_row(row):
    rows = []
    rid = clean_text(row.get('review_id', ''))
    value_text = clean_text(row.get('verified_value', ''))
    metric_family = clean_text(row.get('metric_family', ''))
    phase = clean_text(row.get('phase', ''))
    definition = clean_text(row.get('verified_metric_definition', ''))
    all_text = ' '.join([value_text, definition, clean_text(row.get('notes', ''))]).lower()

    # Known multi-category comparator row.
    if rid == 'cmp_metric_0008':
        add_fig_row(rows, row, 'jobs_per_usd_million', 'Renewable energy', 7.49, split_method='known_multi_value_rule')
        add_fig_row(rows, row, 'jobs_per_usd_million', 'Energy efficiency', 7.72, split_method='known_multi_value_rule')
        add_fig_row(rows, row, 'jobs_per_usd_million', 'Fossil fuels', 2.65, split_method='known_multi_value_rule')
        return rows

    if rid == 'cmp_metric_0012':
        add_fig_row(rows, row, 'jobs_per_usd_million', 'Computer/electronics supply chain for wind', 4.74, split_method='known_supply_chain_rule')
        add_fig_row(rows, row, 'jobs_per_usd_million', 'Construction supply chain benchmark in same table', 8.21, split_method='known_supply_chain_rule', derived=True)
        return rows

    if rid == 'cmp_metric_0013':
        add_fig_row(rows, row, 'jobs_per_usd_million', 'Computer/electronics supply chain for wind', 4.74, split_method='duplicate_of_cmp_metric_0012', plot_ready=False, figure_use='duplicate_reference')
        return rows

    if rid == 'cmp_metric_0014':
        add_fig_row(rows, row, 'absolute_jobs', 'Geothermal deployment: jobs', 10000, split_method='known_geothermal_rule', plot_ready=False, figure_use='absolute_context')
        add_fig_row(rows, row, 'absolute_person_years', 'Geothermal deployment: construction/manufacturing person-years', 36000, split_method='known_geothermal_rule', plot_ready=False, figure_use='absolute_context')
        add_fig_row(rows, row, 'job_years_per_mw', 'Geothermal derived person-years per MW', round(36000/5600, 3), split_method='derived_from_5600mw_and_36000_person_years', derived=True)
        return rows

    if rid == 'dc_metric_0026':
        # 1 job per USD 13 million investment -> 0.0769 jobs per USD 1 million.
        add_fig_row(rows, row, 'jobs_per_usd_million', 'Data centres: permanent jobs per investment (FWW)', round(1/13, 4), split_method='derived_from_1_job_per_13m', derived=True)
        return rows

    if rid == 'dc_metric_0003':
        rng = parse_range(value_text)
        if rng:
            low, high, mid = rng
            add_fig_row(rows, row, 'jobs_per_mw', 'Duplicate of Hamm construction workers per MW range', mid, low, high, split_method='duplicate_of_dc_metric_0001', plot_ready=False, figure_use='duplicate_reference')
            return rows

    # General range rows such as 0.7-2.0, 0.15-0.35, 30-100.
    rng = parse_range(value_text)
    if rng:
        low, high, mid = rng
        if metric_family in ['jobs_per_mw', 'construction_jobs_headcount'] and ('operational' in all_text or 'fte' in all_text or 'o&m' in all_text) and 'construction workers' not in all_text:
            unit = 'operational_fte_per_mw'
        elif metric_family in ['jobs_per_mw', 'construction_jobs_headcount'] and ('per mw' in all_text or '/mw' in all_text or 'workers per mw' in all_text):
            unit = 'jobs_per_mw'
        elif metric_family == 'operational_jobs_fte':
            unit = 'absolute_jobs'
        else:
            unit = 'absolute_jobs' if 'jobs' in metric_family else metric_family
        add_fig_row(rows, row, unit, short_text(definition or value_text, 90), mid, low, high, split_method='range_parse')
        return rows

    # Single-value defaults.
    value = first_numeric_or_none(value_text)
    if value is None:
        add_fig_row(rows, row, 'unparsed', 'Unparsed value', None, split_method='unparsed', plot_ready=False, figure_use='needs_manual_cleaning')
        return rows

    if rid == 'cmp_metric_0019':
        unit = 'job_years_per_mw'
    elif rid == 'cmp_metric_0020':
        unit = 'operational_fte_per_mw'
    elif metric_family == 'jobs_per_mw':
        if 'job-year' in all_text or 'manyear' in all_text:
            unit = 'job_years_per_mw'
        elif phase == 'operation' or 'o&m' in all_text or 'operational' in all_text:
            unit = 'operational_fte_per_mw'
        else:
            unit = 'jobs_per_mw'
    elif metric_family == 'jobs_per_investment':
        unit = 'jobs_per_usd_million'
    elif metric_family == 'job_years_per_gwh':
        unit = 'job_years_per_gwh'
    elif metric_family == 'construction_or_lifecycle_job_years':
        unit = 'absolute_person_years' if 'person-year' in all_text else 'absolute_jobs'
    elif metric_family == 'construction_jobs_headcount':
        unit = 'jobs_per_mw' if ('per mw' in all_text or '/mw' in all_text) else 'absolute_jobs'
    elif metric_family == 'operational_jobs_fte':
        unit = 'absolute_jobs'
    elif metric_family == 'causal_local_economy_effect':
        unit = 'percent_effect'
    else:
        unit = 'other_numeric_context'

    label = short_text(definition or value_text, 90)
    plot_ready = unit in ['jobs_per_mw', 'operational_fte_per_mw', 'jobs_per_usd_million', 'job_years_per_gwh', 'job_years_per_mw', 'percent_effect']
    if unit == 'percent_effect' and 'capacity or % effect per source context' in all_text:
        plot_ready = False
    figure_use = 'plot_candidate' if plot_ready else 'context_only'
    add_fig_row(rows, row, unit, label, value, split_method='single_numeric_parse', plot_ready=plot_ready, figure_use=figure_use)
    return rows

fig_rows = []
for _, row in jobs.iterrows():
    fig_rows.extend(split_job_row(row))

figure_data_raw = pd.DataFrame(fig_rows)

# Collapse exact repeated extractions, especially Hamm project-example duplicates and duplicate comparator framing.
key_cols = ['domain', 'source_id', 'figure_panel', 'figure_unit', 'series_label', 'phase', 'plot_value', 'plot_value_low', 'plot_value_high']
for col in key_cols:
    if col not in figure_data_raw.columns:
        figure_data_raw[col] = ''

agg = {col: 'first' for col in figure_data_raw.columns if col not in key_cols + ['source_evidence_id', 'review_id', 'figure_row_id']}
agg['source_evidence_id'] = lambda s: '; '.join(sorted(set(map(str, s))))
agg['review_id'] = lambda s: '; '.join(sorted(set(map(str, s))))
figure_data = figure_data_raw.groupby(key_cols, dropna=False, as_index=False).agg(agg)
figure_data.insert(0, 'figure_row_id', [f'fig_{i:03d}' for i in range(1, len(figure_data) + 1)])
figure_data['collapsed_duplicate_count'] = figure_data['review_id'].map(lambda x: len([p for p in str(x).split('; ') if p]))

# Order for easier inspection.
panel_order = {
    'A. Jobs or job-years per MW': 1,
    'B. Jobs per USD 1 million investment/spending': 2,
    'C. Job-years per GWh': 3,
    'D. Causal/local percentage effects': 4,
    'E. Absolute/context benchmarks': 5,
}
figure_data['_panel_order'] = figure_data['figure_panel'].map(panel_order).fillna(99)
figure_data = figure_data.sort_values(['_panel_order', 'domain', 'phase', 'plot_value']).drop(columns=['_panel_order']).reset_index(drop=True)
figure_data['figure_row_id'] = [f'fig_{i:03d}' for i in range(1, len(figure_data) + 1)]

print('Raw split figure rows:', len(figure_data_raw))
print('Clean/collapsed figure rows:', len(figure_data))
display(figure_data[['figure_row_id','figure_panel','domain_label','series_label','figure_unit','plot_value','plot_value_low','plot_value_high','range_flag','derived_flag','plot_ready','review_id']].head(30))


In [ ]:
scale_out = scale.copy()

def scale_text_use(row):
    family = clean_text(row.get('metric_family', ''))
    if family == 'investment_scale':
        return 'Investment scale/context only; do not infer jobs without a job-intensity metric.'
    if family == 'capacity_scale':
        return 'Capacity/pipeline context; useful for scaling assumptions after source spot-check.'
    if family == 'construction_duration':
        return 'Timing/skills context; useful for discussing temporary construction demand.'
    return 'Context only.'

scale_context_for_text = pd.DataFrame({
    'scale_row_id': [f'scale_{i:03d}' for i in range(1, len(scale_out) + 1)],
    'evidence_id': scale_out['evidence_id'],
    'domain': scale_out['domain'].map(domain_label),
    'asset_or_comparator': scale_out['asset_class'],
    'metric_family': scale_out['metric_family'],
    'phase': scale_out['phase'],
    'verified_value': scale_out['verified_value'],
    'metric_definition': scale_out['verified_metric_definition'].map(lambda x: short_text(x, 320)),
    'geography': scale_out['geography'],
    'currency_price_year': scale_out['currency_price_year'],
    'source': scale_out.apply(source_label, axis=1),
    'review_decision': scale_out['review_decision'],
    'evidence_role': scale_out['evidence_role'],
    'text_use': scale_out.apply(scale_text_use, axis=1),
    'notes': scale_out['notes'].map(lambda x: short_text(x, 360)),
    'source_url': scale_out['source_url'],
    'source_raw_path': scale_out['source_raw_path'],
})

print('Scale context rows:', len(scale_context_for_text))
display(scale_context_for_text.head(10))


In [ ]:
spotcheck_queue = table1_out[table1_out['spotcheck_required'].astype(str).str.lower().isin(['true', '1', 'yes'])].copy()
spotcheck_queue.insert(0, 'spotcheck_id', [f'check_{i:03d}' for i in range(1, len(spotcheck_queue) + 1)])
spotcheck_queue['spotcheck_priority'] = spotcheck_queue['evidence_role'].map(
    lambda x: 'high' if x == 'primary_citable_after_spotcheck' else 'medium'
)
spotcheck_queue['spotcheck_instruction'] = 'Verify page/location, value, unit, direct-vs-total scope, and caveat against original source before manuscript citation.'

caption_notes = pd.DataFrame([
    {
        'caption_id': 'cap_001',
        'target': 'Table 1',
        'draft_caption_or_note': 'Reviewed evidence on data-centre job creation and comparator asset benchmarks. Rows retain scope, phase, geography, and caveats because direct jobs, total supported jobs, job-years, and causal effects are not interchangeable.',
        'caution': 'Do not average unlike units; keep evidence_role and caveat_summary visible during drafting.',
    },
    {
        'caption_id': 'cap_002',
        'target': 'Figure - jobs per MW',
        'draft_caption_or_note': 'Indicative job intensity per MW distinguishes temporary construction demand from sustained operational staffing. Data-centre construction examples can be labour-intensive at peak, while long-term operations are much lower per MW.',
        'caution': 'Ranges and project examples are not population averages; use only after source spot-check.',
    },
    {
        'caption_id': 'cap_003',
        'target': 'Figure - jobs per USD 1 million',
        'draft_caption_or_note': 'Jobs per investment benchmarks place data-centre permanent job yield beside energy and efficiency comparators.',
        'caution': 'Scopes differ: data-centre rows may be direct permanent jobs, while comparator I-O rows are often total direct plus indirect plus induced FTE.',
    },
    {
        'caption_id': 'cap_004',
        'target': 'Scale context',
        'draft_caption_or_note': 'Investment, capacity, and construction-duration rows provide scale context for interpreting job metrics.',
        'caution': 'Scale rows are not job-creation evidence unless paired with a valid job-intensity metric.',
    },
])

print('Spotcheck queue rows:', len(spotcheck_queue))
display(caption_notes)


In [ ]:
out_table1 = OUTPUT_DIR / 'table_1_data_centre_vs_comparators.csv'
out_figure = OUTPUT_DIR / 'figure_job_intensity_clean_data.csv'
out_scale = OUTPUT_DIR / 'scale_context_for_text.csv'
out_spotcheck = OUTPUT_DIR / 'source_spotcheck_queue.csv'
out_caption = OUTPUT_DIR / 'figure_caption_notes.csv'
out_xlsx = OUTPUT_DIR / 'manuscript_primary_evidence_table.xlsx'

# Save CSVs.
table1_out.to_csv(out_table1, index=False)
figure_data.to_csv(out_figure, index=False)
scale_context_for_text.to_csv(out_scale, index=False)
spotcheck_queue.to_csv(out_spotcheck, index=False)
caption_notes.to_csv(out_caption, index=False)

def format_worksheet(ws):
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions
    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = copy(cell.alignment)
            cell.alignment = cell.alignment.copy(wrap_text=True, vertical='top')
    for col in ws.columns:
        header = str(col[0].value or '')
        width = min(max(len(header) + 2, 12), 46)
        for cell in col[1:80]:
            width = min(max(width, min(len(str(cell.value or '')) + 2, 46)), 46)
        ws.column_dimensions[col[0].column_letter].width = width

readme = pd.DataFrame({
    'item': [
        'created', 'basis', 'table_1_rows', 'figure_rows_clean', 'figure_rows_plot_ready',
        'scale_context_rows', 'spotcheck_rows', 'main_warning'
    ],
    'value': [
        RUN_DATE,
        'Notebook 06 reviewed comparative evidence outputs only',
        len(table1_out),
        len(figure_data),
        int(figure_data['plot_ready'].astype(bool).sum()),
        len(scale_context_for_text),
        len(spotcheck_queue),
        'Keep direct_vs_total, projected_vs_realised, evidence_role, and caveats visible. Do not average unlike units.'
    ]
})

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    readme.to_excel(writer, sheet_name='README', index=False)
    table1_out.to_excel(writer, sheet_name='Table1_Evidence', index=False)
    figure_data.to_excel(writer, sheet_name='FigureData_Clean', index=False)
    scale_context_for_text.to_excel(writer, sheet_name='ScaleContext_Text', index=False)
    spotcheck_queue.to_excel(writer, sheet_name='SpotcheckQueue', index=False)
    caption_notes.to_excel(writer, sheet_name='CaptionNotes', index=False)
    gap_log.to_excel(writer, sheet_name='GapLog_From06', index=False)
    for ws in writer.book.worksheets:
        format_worksheet(ws)

print('Wrote:')
for path in [out_xlsx, out_table1, out_figure, out_scale, out_spotcheck, out_caption]:
    print('-', path)


In [ ]:
notes_md = OUTPUT_DIR / 'MANUSCRIPT_TABLES_FIGURES_NOTES.md'
plot_ready_count = int(figure_data['plot_ready'].astype(bool).sum())
range_count = int(figure_data['range_flag'].astype(bool).sum())
derived_count = int(figure_data['derived_flag'].astype(bool).sum())

notes = f'''# Manuscript Tables and Figure Data Notes

Date: {RUN_DATE}

## Inputs

This notebook uses only outputs from notebook 06:

- `comparative_evidence_primary_table.csv`
- `jobs_intensity_comparison_for_figures.csv`
- `scale_context_table.csv`
- `evidence_gap_log.csv`

No new search, download, source extraction, or evidence review is performed here.

## Outputs

- `manuscript_primary_evidence_table.xlsx`: workbook with table, figure-data, scale-context, spot-check, caption-note, and gap-log sheets.
- `table_1_data_centre_vs_comparators.csv`: compact manuscript evidence table ({len(table1_out):,} rows).
- `figure_job_intensity_clean_data.csv`: split and cleaned figure dataset ({len(figure_data):,} rows; {plot_ready_count:,} plot-ready rows).
- `scale_context_for_text.csv`: investment, capacity, and timing context for prose ({len(scale_context_for_text):,} rows).
- `source_spotcheck_queue.csv`: rows requiring final source/PDF checks before citation ({len(spotcheck_queue):,} rows).
- `figure_caption_notes.csv`: draft caption notes and cautions.

## Figure Data Status

The figure dataset splits key multi-value evidence where possible. It includes {range_count:,} range rows and {derived_count:,} derived rows. Derived rows are explicitly marked in `derived_flag` and should be used only if the derivation is acceptable for the manuscript.

Recommended plotting order:

1. Start with `figure_panel = A. Jobs or job-years per MW` to compare data-centre construction/operation with wind/geothermal-style benchmarks.
2. Use `figure_panel = B. Jobs per USD 1 million investment/spending` to compare data-centre permanent job yield with energy/efficiency investment multipliers.
3. Treat causal percentage-effect rows and absolute headcount rows as context panels or text evidence, not as directly comparable intensities.

## Use Rules

- Keep `direct_vs_total`, `projected_vs_realised`, `evidence_role`, and `notes` visible in any manuscript table draft.
- Do not average data-centre direct FTE rows with comparator total I-O multiplier rows.
- Do not convert scale/capacity/investment rows into job claims unless paired with a defensible intensity metric.
- Final manuscript citations still require original source/PDF spot-checks.

## Remaining Gap

The same gap from notebook 06 remains: high-rise or commercial building comparator evidence is still thin relative to energy, infrastructure, and advanced manufacturing comparators. A targeted search may be needed if the article wants a building-sector comparator rather than a broader asset-class comparator.
'''
notes_md.write_text(notes, encoding='utf-8')
print(notes_md)
print(notes)


In [ ]:
# Compact QA checks.
assert len(table1_out) == len(primary), 'Table 1 should preserve primary evidence row count.'
assert table1_out['evidence_id'].is_unique, 'Table 1 evidence IDs are not unique.'
assert not figure_data['figure_row_id'].duplicated().any(), 'Figure row IDs are not unique.'
assert figure_data['plot_value'].notna().any(), 'No numeric figure values were created.'
assert out_xlsx.exists(), 'Workbook was not written.'

print('QA passed.')
print('Output workbook:', out_xlsx)
print('Table 1 rows:', len(table1_out))
print('Clean figure rows:', len(figure_data), '| plot-ready:', int(figure_data['plot_ready'].astype(bool).sum()))
print('Scale context rows:', len(scale_context_for_text))
print('Spotcheck queue rows:', len(spotcheck_queue))
